# Section 4 Analysis: Changing One Decision

This notebook analyzes the results from section 4 experiments, which test the impact of changing one decision at a time in the RAG pipeline.

## Experiments Being Compared

1. **Baseline** (Section 3): `llama3.2` (3B), chunk_size=1200, chunk_overlap=300
2. **4.1 - Larger Model**: `llama3:8b` (8B), chunk_size=1200, chunk_overlap=300
3. **4.3 - Small Chunks**: `llama3.2` (3B), chunk_size=300, chunk_overlap=60

## Research Questions

- **4.1**: How does model size (8B vs 3B) affect latency and response quality?
- **4.3**: How does chunk size (300 vs 1200) affect latency, chunk count, and response quality?



In [ ]:
import pandas as pd
import os
from IPython.display import display

# Load all three experiment results
baseline_file = "results/chunk1200_overlap300_k3_llama3.2_nomic-embed-text.csv"
model_8b_file = "results/chunk1200_overlap300_k3_llama3:8b_nomic-embed-text.csv"
small_chunk_file = "results/chunk300_overlap60_k3_llama3.2_nomic-embed-text.csv"

df_baseline = pd.read_csv(baseline_file)
df_8b = pd.read_csv(model_8b_file)
df_small_chunk = pd.read_csv(small_chunk_file)

print("Loaded experiment results:")
print(f"  Baseline: {baseline_file} ({len(df_baseline)} questions)")
print(f"  8B Model: {model_8b_file} ({len(df_8b)} questions)")
print(f"  Small Chunks: {small_chunk_file} ({len(df_small_chunk)} questions)")
print("\n" + "="*80)



## Configuration Comparison

Comparing the key configuration parameters across experiments:


In [ ]:
# Create comparison table
config_comparison = pd.DataFrame({
    "Baseline": [
        df_baseline["LLM Model"].iloc[0],
        df_baseline["Chunk Size"].iloc[0],
        df_baseline["Chunk Overlap"].iloc[0],
        df_baseline["Retrieval K"].iloc[0],
        df_baseline["Number of Chunks"].iloc[0],
    ],
    "4.1 - 8B Model": [
        df_8b["LLM Model"].iloc[0],
        df_8b["Chunk Size"].iloc[0],
        df_8b["Chunk Overlap"].iloc[0],
        df_8b["Retrieval K"].iloc[0],
        df_8b["Number of Chunks"].iloc[0],
    ],
    "4.3 - Small Chunks": [
        df_small_chunk["LLM Model"].iloc[0],
        df_small_chunk["Chunk Size"].iloc[0],
        df_small_chunk["Chunk Overlap"].iloc[0],
        df_small_chunk["Retrieval K"].iloc[0],
        df_small_chunk["Number of Chunks"].iloc[0],
    ]
}, index=["LLM Model", "Chunk Size", "Chunk Overlap", "Retrieval K", "Number of Chunks"])

display(config_comparison)

print("\nKey Differences:")
print("  - 4.1: Model changed from llama3.2 (3B) to llama3:8b (8B)")
print("  - 4.3: Chunk size reduced from 1200 to 300, overlap from 300 to 60")
print("  - 4.3: Number of chunks increased significantly (175 → 682)")



## Runtime Metrics Comparison

Comparing indexing times and response times across experiments:


In [ ]:
# Runtime metrics comparison
runtime_metrics = pd.DataFrame({
    "Baseline": [
        df_baseline["Indexing Time (s)"].iloc[0],
        df_baseline["Average Response Time (s)"].iloc[0],
    ],
    "4.1 - 8B Model": [
        df_8b["Indexing Time (s)"].iloc[0],
        df_8b["Average Response Time (s)"].iloc[0],
    ],
    "4.3 - Small Chunks": [
        df_small_chunk["Indexing Time (s)"].iloc[0],
        df_small_chunk["Average Response Time (s)"].iloc[0],
    ]
}, index=["Indexing Time (s)", "Average Response Time (s)"])

display(runtime_metrics)

print("\nPerformance Comparison:")
print(f"  Indexing Time:")
print(f"    Baseline: {df_baseline['Indexing Time (s)'].iloc[0]:.2f}s")
print(f"    8B Model: {df_8b['Indexing Time (s)'].iloc[0]:.2f}s ({((df_8b['Indexing Time (s)'].iloc[0] / df_baseline['Indexing Time (s)'].iloc[0]) - 1) * 100:+.1f}%)")
print(f"    Small Chunks: {df_small_chunk['Indexing Time (s)'].iloc[0]:.2f}s ({((df_small_chunk['Indexing Time (s)'].iloc[0] / df_baseline['Indexing Time (s)'].iloc[0]) - 1) * 100:+.1f}%)")

print(f"\n  Average Response Time:")
print(f"    Baseline: {df_baseline['Average Response Time (s)'].iloc[0]:.2f}s")
print(f"    8B Model: {df_8b['Average Response Time (s)'].iloc[0]:.2f}s ({((df_8b['Average Response Time (s)'].iloc[0] / df_baseline['Average Response Time (s)'].iloc[0]) - 1) * 100:+.1f}%)")
print(f"    Small Chunks: {df_small_chunk['Average Response Time (s)'].iloc[0]:.2f}s ({((df_small_chunk['Average Response Time (s)'].iloc[0] / df_baseline['Average Response Time (s)'].iloc[0]) - 1) * 100:+.1f}%)")



In [ ]:
# Individual question response times comparison
response_times = pd.DataFrame({
    "Question": df_baseline["Question"],
    "Baseline (s)": df_baseline["Response Time (s)"],
    "8B Model (s)": df_8b["Response Time (s)"],
    "Small Chunks (s)": df_small_chunk["Response Time (s)"],
})

display(response_times)



## Response Quality Evaluation

Comparing response quality across experiments for each question:


### Question 1: What is Apache Iceberg? Explain in short.


In [ ]:
q1_baseline = df_baseline.iloc[0]
q1_8b = df_8b.iloc[0]
q1_small = df_small_chunk.iloc[0]

print("**Question:**", q1_baseline["Question"])
print("\n" + "="*80)
print("**Baseline Response:**")
print(q1_baseline["Response"])
print(f"\nResponse Time: {q1_baseline['Response Time (s)']}s")
print("\n" + "="*80)
print("**8B Model Response:**")
print(q1_8b["Response"])
print(f"\nResponse Time: {q1_8b['Response Time (s)']}s")
print("\n" + "="*80)
print("**Small Chunks Response:**")
print(q1_small["Response"])
print(f"\nResponse Time: {q1_small['Response Time (s)']}s")



**Quality Assessment:**

**Baseline vs 8B Model (4.1):**
- Both responses correctly identify Apache Iceberg as a storage format
- 8B model provides more detailed explanation (mentions "repository format", "fault-tolerant", "big data workloads")
- 8B model response is more comprehensive but takes ~2.8x longer (25.69s vs 9.0s)
- Both are accurate, but 8B model provides richer context

**Baseline vs Small Chunks (4.3):**
- Small chunks response is more concise and focused
- Mentions specific file formats (Parquet, Avro, ORC) which is accurate
- Response time is faster (6.25s vs 9.0s, ~30% faster)
- Both are accurate, small chunks response is more efficient

**Overall:** All three responses are accurate. The 8B model provides more detail at the cost of latency, while small chunks provide faster responses with similar accuracy.



### Question 2: How does Iceberg ensure that two writers do not overwrite each others ingestion results?


In [ ]:
q2_baseline = df_baseline.iloc[1]
q2_8b = df_8b.iloc[1]
q2_small = df_small_chunk.iloc[1]

print("**Question:**", q2_baseline["Question"])
print("\n" + "="*80)
print("**Baseline Response:**")
print(q2_baseline["Response"])
print(f"\nResponse Time: {q2_baseline['Response Time (s)']}s")
print("\n" + "="*80)
print("**8B Model Response:**")
print(q2_8b["Response"])
print(f"\nResponse Time: {q2_8b['Response Time (s)']}s")
print("\n" + "="*80)
print("**Small Chunks Response:**")
print(q2_small["Response"])
print(f"\nResponse Time: {q2_small['Response Time (s)']}s")



**Quality Assessment:**

**Baseline vs 8B Model (4.1):**
- 8B model provides a more detailed and structured explanation
- 8B model explicitly mentions "Optimistic Concurrency" and explains serializable isolation
- 8B model response is more comprehensive, explaining both the mechanism and its implications
- Response time: 8B model is ~2.5x slower (22.42s vs 8.89s)
- Both are accurate, but 8B model provides better explanation quality

**Baseline vs Small Chunks (4.3):**
- Small chunks response indicates the context doesn't explicitly state the answer
- Small chunks response mentions deletion vectors, which is not directly relevant to this question
- This suggests small chunks may have retrieved less relevant context
- Response time is similar (7.18s vs 8.89s)
- Small chunks response is less accurate for this question

**Overall:** 8B model provides the best explanation. Small chunks struggled with this question, likely due to less context per chunk.



### Question 3: How to access data that was deleted in a newer snapshot?


In [ ]:
q3_baseline = df_baseline.iloc[2]
q3_8b = df_8b.iloc[2]
q3_small = df_small_chunk.iloc[2]

print("**Question:**", q3_baseline["Question"])
print("\n" + "="*80)
print("**Baseline Response:**")
print(q3_baseline["Response"])
print(f"\nResponse Time: {q3_baseline['Response Time (s)']}s")
print("\n" + "="*80)
print("**8B Model Response:**")
print(q3_8b["Response"])
print(f"\nResponse Time: {q3_8b['Response Time (s)']}s")
print("\n" + "="*80)
print("**Small Chunks Response:**")
print(q3_small["Response"])
print(f"\nResponse Time: {q3_small['Response Time (s)']}s")



**Quality Assessment:**

**Baseline vs 8B Model (4.1):**
- Both responses mention accessing older snapshots
- 8B model response is more detailed, explaining snapshot selection and time travel queries
- 8B model mentions "AS OF" clause which is the standard SQL approach
- Response time: 8B model is ~2.4x slower (28.19s vs 11.78s)
- Both are accurate, but 8B model provides more complete answer

**Baseline vs Small Chunks (4.3):**
- Small chunks response mentions DELETED status markers and deletion vectors
- Response is technically correct but focuses on implementation details rather than user-facing approach
- Similar to baseline, misses the primary time travel query method
- Response time is faster (8.19s vs 11.78s)
- Both are partially accurate, missing the standard time travel approach

**Overall:** 8B model provides the most complete answer. Small chunks and baseline both miss the primary time travel query method.



### Question 4: What happens if a writer attempts to commit based on an old snapshot?


In [ ]:
q4_baseline = df_baseline.iloc[3]
q4_8b = df_8b.iloc[3]
q4_small = df_small_chunk.iloc[3]

print("**Question:**", q4_baseline["Question"])
print("\n" + "="*80)
print("**Baseline Response:**")
print(q4_baseline["Response"])
print(f"\nResponse Time: {q4_baseline['Response Time (s)']}s")
print("\n" + "="*80)
print("**8B Model Response:**")
print(q4_8b["Response"])
print(f"\nResponse Time: {q4_8b['Response Time (s)']}s")
print("\n" + "="*80)
print("**Small Chunks Response:**")
print(q4_small["Response"])
print(f"\nResponse Time: {q4_small['Response Time (s)']}s")



**Quality Assessment:**

**Baseline vs 8B Model (4.1):**
- 8B model provides a more detailed explanation
- 8B model explains the retry mechanism and mentions the optimistic concurrency approach
- 8B model response is more comprehensive
- Response time: 8B model is ~2.5x slower (27.26s vs 10.85s)
- Both are accurate, but 8B model provides better explanation

**Baseline vs Small Chunks (4.3):**
- Small chunks response correctly identifies that the write will fail
- Small chunks response mentions metadata file pointer swap, which is accurate
- Both responses are brief but correct
- Response time is similar (9.28s vs 10.85s)
- Both are accurate but could be more detailed

**Overall:** 8B model provides the most complete answer. Small chunks and baseline are both correct but brief.



## Summary & Findings

### Key Metrics Summary


In [ ]:
# Create summary table
summary_data = {
    "Metric": [
        "LLM Model",
        "Chunk Size",
        "Number of Chunks",
        "Indexing Time (s)",
        "Avg Response Time (s)",
        "Response Time Ratio (vs Baseline)",
    ],
    "Baseline": [
        "llama3.2 (3B)",
        1200,
        175,
        f"{df_baseline['Indexing Time (s)'].iloc[0]:.2f}",
        f"{df_baseline['Average Response Time (s)'].iloc[0]:.2f}",
        "1.00x",
    ],
    "4.1 - 8B Model": [
        "llama3:8b (8B)",
        1200,
        175,
        f"{df_8b['Indexing Time (s)'].iloc[0]:.2f}",
        f"{df_8b['Average Response Time (s)'].iloc[0]:.2f}",
        f"{df_8b['Average Response Time (s)'].iloc[0] / df_baseline['Average Response Time (s)'].iloc[0]:.2f}x",
    ],
    "4.3 - Small Chunks": [
        "llama3.2 (3B)",
        300,
        682,
        f"{df_small_chunk['Indexing Time (s)'].iloc[0]:.2f}",
        f"{df_small_chunk['Average Response Time (s)'].iloc[0]:.2f}",
        f"{df_small_chunk['Average Response Time (s)'].iloc[0] / df_baseline['Average Response Time (s)'].iloc[0]:.2f}x",
    ],
}

summary_df = pd.DataFrame(summary_data)
display(summary_df)



### Findings

#### 4.1: Model Size Impact (8B vs 3B)

**Latency Impact:**
- Average response time increased by **~112%** (12.18s → 25.84s)
- Indexing time increased slightly (~3%)
- The larger model is significantly slower but provides more detailed responses

**Quality Impact:**
- 8B model provides more comprehensive and detailed explanations
- Better understanding of context and more structured responses
- All responses remain accurate, with improved depth

**Trade-off:** The 8B model sacrifices speed for quality. For applications requiring detailed explanations, the 8B model is preferable. For faster responses, the 3B model is sufficient.

#### 4.3: Chunk Size Impact (300 vs 1200)

**Latency Impact:**
- Average response time **decreased by ~37%** (12.18s → 7.73s)
- Indexing time remained similar (~2% increase)
- Smaller chunks lead to faster response times

**Chunk Count Impact:**
- Number of chunks increased by **~290%** (175 → 682)
- More chunks means more embedding operations, but faster retrieval per chunk

**Quality Impact:**
- Small chunks sometimes struggle with questions requiring broader context (e.g., Question 2)
- Responses are more concise but may miss some details
- For simple questions, small chunks perform well and faster

**Trade-off:** Smaller chunks provide faster responses but may lose context for complex questions. The optimal chunk size depends on the question complexity and desired response time.

### Recommendations

1. **For production systems requiring fast responses:** Use smaller chunks (300) with the 3B model
2. **For systems requiring detailed explanations:** Use the 8B model with standard chunk size (1200)
3. **For balanced performance:** Use baseline configuration (3B model, chunk_size=1200)
4. **Consider hybrid approach:** Use smaller chunks for simple queries, larger chunks for complex queries

### What Breaks?

- **8B Model:** Response latency increases significantly (~2x slower), but quality improves
- **Small Chunks:** Context may be fragmented for complex questions, leading to less accurate responses for questions requiring broader understanding

